Clone the public repo to get the data

In [ ]:
!git clone https://github.com/jacobawinter/question_classification.git


fatal: destination path 'question_classification' already exists and is not an empty directory.


In [ ]:
"Hello"

In [ ]:
%pip install transformers datasets scikit-learn torch

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score, cohen_kappa_score
from google.colab import drive

# ── Config ────────────────────────────────────────────────────────────────────

drive.mount('/content/drive')

MODELS = [
    "distilbert-base-uncased",
   # "roberta-base",
    "microsoft/deberta-v3-base",
]

MAX_LEN          = 128   # 4 sentences fits comfortably
BATCH_SIZE       = 16    # safe for free Colab T4
ACCUMULATION     = 2     # simulates batch size of 32
EPOCHS           = 3
LR               = 2e-5
N_FOLDS          = 3
RANDOM_STATE     = 12321
DEVICE           = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR   = "/content/drive/MyDrive/checkpoints"   # saves to Drive so disconnects don't lose progress
RESULTS_PATH     = f"{CHECKPOINT_DIR}/results.json"


os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Using device: {DEVICE}")
print(f"Checkpoints -> {CHECKPOINT_DIR}")

# ── Dataset ───────────────────────────────────────────────────────────────────
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx]
        }

# ── Train / Evaluate helpers ──────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    for i, batch in enumerate(loader):
        outputs = model(
            input_ids      = batch["input_ids"].to(DEVICE),
            attention_mask = batch["attention_mask"].to(DEVICE),
            labels         = batch["labels"].to(DEVICE)
        )
        loss = outputs.loss / ACCUMULATION
        loss.backward()
        total_loss += outputs.loss.item()

        if (i + 1) % ACCUMULATION == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    return total_loss / len(loader)


def evaluate_model(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            outputs = model(
                input_ids      = batch["input_ids"].to(DEVICE),
                attention_mask = batch["attention_mask"].to(DEVICE)
            )
            preds.extend(outputs.logits.argmax(-1).cpu().numpy())
            targets.extend(batch["labels"].numpy())
    return np.array(preds), np.array(targets)


def save_checkpoint(model_name, cv_scores, metrics=None):
    """Save progress to Drive after each fold and model completes."""
    safe_name   = model_name.replace("/", "_")
    ckpt_path   = f"{CHECKPOINT_DIR}/{safe_name}_progress.json"
    data        = {"model_name": model_name, "cv_scores": cv_scores, "metrics": metrics}
    with open(ckpt_path, "w") as f:
        json.dump(data, f)
    print(f"  ✓ Checkpoint saved -> {ckpt_path}")


def load_checkpoint(model_name):
    """Resume from a previous run if checkpoint exists."""
    safe_name = model_name.replace("/", "_")
    ckpt_path = f"{CHECKPOINT_DIR}/{safe_name}_progress.json"
    if os.path.exists(ckpt_path):
        with open(ckpt_path) as f:
            data = json.load(f)
        print(f"  ↩ Resuming {model_name} — {len(data['cv_scores'])} folds already done")
        return data["cv_scores"], data["metrics"]
    return [], None


def save_results(results):
    """Persist results table to Drive after each model completes."""
    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f)
    print(f"  ✓ Results saved -> {RESULTS_PATH}")


def load_results():
    """Load any previously completed model results."""
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH) as f:
            results = json.load(f)
        print(f"  ↩ Found {len(results)} completed model(s) in results checkpoint")
        return results
    return []


# ── Per-model run ─────────────────────────────────────────────────────────────
def run_model(model_name, train, test, num_labels):
    print(f"Model: {model_name}")
    print(f"{'═'*60}")

    # Resume if this model was interrupted
    cv_scores, completed_metrics = load_checkpoint(model_name)
    if completed_metrics is not None:
        print(f"  Model fully complete — skipping.")
        return completed_metrics

    tokenizer    = AutoTokenizer.from_pretrained(model_name)
    skf          = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    folds        = list(skf.split(train["text"], train["label"]))
    start_fold   = len(cv_scores)  # skip already-completed folds

    # ── CV on training data ───────────────────────────────────────────────────
    for fold, (train_idx, val_idx) in enumerate(folds):
        if fold < start_fold:
            print(f"\n  Fold {fold + 1} / {N_FOLDS}  (already done, skipping)")
            continue

        print(f"\n  Fold {fold + 1} / {N_FOLDS}")
        fold_train   = train.iloc[train_idx]
        fold_val     = train.iloc[val_idx]
        train_loader = DataLoader(TextDataset(fold_train["text"], fold_train["label"], tokenizer), batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(TextDataset(fold_val["text"],   fold_val["label"],   tokenizer), batch_size=BATCH_SIZE)

        model        = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels).to(DEVICE)
        total_steps  = (len(train_loader) // ACCUMULATION) * EPOCHS
        optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
        scheduler    = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

        for epoch in range(EPOCHS):
            loss = train_epoch(model, train_loader, optimizer, scheduler)
            print(f"    Epoch {epoch + 1}  loss={loss:.4f}")

        preds, targets = evaluate_model(model, val_loader)
        fold_f1        = f1_score(targets, preds, average="macro")
        cv_scores.append(fold_f1)
        print(f"  Fold F1: {fold_f1:.4f}")

        # Save after every fold
        save_checkpoint(model_name, cv_scores)

        del model
        torch.cuda.empty_cache()

    cv_mean, cv_std = np.mean(cv_scores), np.std(cv_scores)
    print(f"\n  CV F1: {cv_mean:.4f} ± {cv_std:.4f}")

    # ── Retrain on full training data ─────────────────────────────────────────
    print(f"\n  Retraining on full training set...")
    full_loader  = DataLoader(TextDataset(train["text"], train["label"], tokenizer), batch_size=BATCH_SIZE, shuffle=True)
    final_model  = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels).to(DEVICE)
    total_steps  = (len(full_loader) // ACCUMULATION) * EPOCHS
    optimizer    = AdamW(final_model.parameters(), lr=LR, weight_decay=0.01)
    scheduler    = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

    for epoch in range(EPOCHS):
        loss = train_epoch(final_model, full_loader, optimizer, scheduler)
        print(f"    Epoch {epoch + 1}  loss={loss:.4f}")

    # Save final model weights to Drive
    safe_name   = model_name.replace("/", "_")
    model_path  = f"{CHECKPOINT_DIR}/{safe_name}_final"
    final_model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    print(f"  ✓ Final model saved -> {model_path}")

    # ── Evaluate on held-out test set ─────────────────────────────────────────
    test_loader = DataLoader(TextDataset(test["text"], test["label"], tokenizer), batch_size=BATCH_SIZE)
    preds, targets = evaluate_model(final_model, test_loader)

    print(f"\n  Test set results:")
    print(classification_report(targets, preds))

    metrics = get_metrics(targets, preds, model_name, cv_mean, cv_std)

    # Mark model as fully complete
    save_checkpoint(model_name, cv_scores, metrics=metrics)

    return metrics


# ── Main ──────────────────────────────────────────────────────────────────────
def run(df: pd.DataFrame):
    num_labels = df["label"].nunique()

    train_df, test_df = train_test_split(
        df,
        test_size    = 0.15,
        stratify     = df["label"],
        random_state = RANDOM_STATE
    )
    print(f"Train: {len(train_df):,} | Test: {len(test_df):,} | Labels: {num_labels}")

    # Resume any previously completed models
    results      = load_results()
    done_models  = {r["model"] for r in results}

    for model_name in MODELS:
        if model_name in done_models:
            print(f"\n  Skipping {model_name} — already in results")
            continue
        row = run_model(model_name, train_df, test_df, num_labels)
        results.append(row)
        save_results(results)  # persist after each model

    results_df = pd.DataFrame(results).sort_values("f1", ascending=False)

    print(f"\n{'═'*60}")
    print("Final comparison:")
    print(results_df.to_string(index=False))

    return results_df


# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    os.chdir('question_classification')
    labelled = pd.read_excel('data/raw/questions_zm_label_sample_0-200.xlsx')
    labelled = labelled[['question_id', 'label_spending_req','label_particular']]
    full = pd.read_csv('data/raw/questions_zm_combined.csv', encoding = 'latin-1')

    df = full.merge(labelled, on = 'question_id', how = 'left')
    df = df.dropna(subset = ['label_spending_req'])


    df['label'] = df['label_spending_req'].str.contains('spend', case = False, na = False)
    #df['label'] = df['label_particular']==1

    df['text'] = df['question']

    results = run(df)
    results.to_csv(f"{CHECKPOINT_DIR}/bert_comparison_results.csv", index=False)

Mounted at /content/drive
Using device: cpu
Checkpoints -> /content/drive/MyDrive/checkpoints
Train: 170 | Test: 30 | Labels: 2
Model: distilbert-base-uncased
════════════════════════════════════════════════════════════


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

    Epoch 1  loss=0.6739
    Epoch 2  loss=0.6163
    Epoch 3  loss=0.5954


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Final model saved -> /content/drive/MyDrive/checkpoints/distilbert-base-uncased_final

  Test set results:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        11
           1       0.63      1.00      0.78        19

    accuracy                           0.63        30
   macro avg       0.32      0.50      0.39        30
weighted avg       0.40      0.63      0.49        30



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


NameError: name 'get_metrics' is not defined